In [1]:
import pandas as pd

file_path = "Reviews.csv"
df = pd.read_csv(file_path, encoding="latin1")

# keep only Score and Text
df = df[['Score', 'Text']]

df['Label'] = df['Score'].apply(lambda x: 'negative' if x in [1, 2] else 'neutral' if x == 3 else 'positive')

print(df.head())

   Score                                               Text     Label
0      5  I have bought several of the Vitality canned d...  positive
1      1  Product arrived labeled as Jumbo Salted Peanut...  negative
2      4  This is a confection that has been around a fe...  positive
3      2  If you are looking for the secret ingredient i...  negative
4      5  Great taffy at a great price.  There was a wid...  positive


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

# Features and labels
texts = df['Text']
labels = df['Label']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.4, random_state=42
)

# Extract features (TF-IDF representation)
vectorizer = TfidfVectorizer(
    stop_words='english',
    lowercase=True,
    token_pattern=r'(?u)\b[a-zA-Z]{2,}\b',
    max_features=5000
)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

# Initialize classifiers
nb_classifier = MultinomialNB()
svm_classifier = LinearSVC()

# Train classifiers
nb_classifier.fit(X_train, y_train)
svm_classifier.fit(X_train, y_train)

# Predict sentiment using classifiers
for text in X_test:
    nb_prediction = nb_classifier.predict(text)[0]
    svm_prediction = svm_classifier.predict(text)[0]

# Calculate classification report for Naive Bayes
nb_classification_report = classification_report(
    y_test,
    nb_classifier.predict(X_test),
    target_names=['negative', 'neutral', 'positive']
)

# Calculate classification report for SVM
svm_classification_report = classification_report(
    y_test,
    svm_classifier.predict(X_test),
    target_names=['negative', 'neutral', 'positive']
)

# Print classification report for Naive Bayes
print("\nClassification Report for Naive Bayes (TF-IDF):")
print(nb_classification_report)
print("Accuracy:", accuracy_score(y_test, nb_classifier.predict(X_test)))

# Print classification report for SVM
print("\nClassification Report for LinearSVC (TF-IDF):")
print(svm_classification_report)
print("Accuracy:", accuracy_score(y_test, svm_classifier.predict(X_test)))


Classification Report for Naive Bayes (TF-IDF):
              precision    recall  f1-score   support

    negative       0.83      0.28      0.41     32531
     neutral       0.44      0.00      0.00     17155
    positive       0.82      0.99      0.90    177696

    accuracy                           0.82    227382
   macro avg       0.70      0.42      0.44    227382
weighted avg       0.79      0.82      0.76    227382

Accuracy: 0.8170347696827366

Classification Report for LinearSVC (TF-IDF):
              precision    recall  f1-score   support

    negative       0.72      0.67      0.70     32531
     neutral       0.60      0.10      0.17     17155
    positive       0.89      0.97      0.93    177696

    accuracy                           0.86    227382
   macro avg       0.74      0.58      0.60    227382
weighted avg       0.84      0.86      0.84    227382

Accuracy: 0.8638194755961334
